# 01 - Extracao: SQL Server para MinIO (CSV)

Este notebook le as quatro tabelas do banco **LojaDB** no SQL Server usando **Spark/JDBC** e grava um arquivo CSV por tabela no bucket **landing-zone** do MinIO.

Fluxo executado:

`SQL Server / LojaDB -> clientes, produtos, pedidos, itens_pedido -> MinIO / landing-zone`

## 1. Configuracao

In [ ]:
import os
import shutil
import tempfile
from pathlib import Path

import boto3
from botocore.client import Config
from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv(override=True)

DB_SERVER = os.getenv('DB_SERVER')
DB_PORT = os.getenv('DB_PORT')
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_DATABASE = os.getenv('DB_DATABASE', 'LojaDB')

MINIO_ENDPOINT = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')

tabelas = ['clientes', 'produtos', 'pedidos', 'itens_pedido']

print(f'SQL Server: {DB_SERVER}:{DB_PORT}/{DB_DATABASE}')
print(f'MinIO: {MINIO_ENDPOINT} | Bucket: {LANDING_BUCKET}')
print(f'Tabelas: {tabelas}')

## 2. Criar SparkSession com driver JDBC do SQL Server

In [ ]:
spark = (
    SparkSession.builder
    .appName('SQLServer_to_MinIO_CSV')
    .master('local[*]')
    .config('spark.jars.packages', 'com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print('SparkSession criada com sucesso.')

## 3. Configurar conexao JDBC

In [ ]:
jdbc_url = (
    f'jdbc:sqlserver://{DB_SERVER}:{DB_PORT};'
    f'databaseName={DB_DATABASE};'
    'encrypt=true;'
    'trustServerCertificate=true;'
)

jdbc_properties = {
    'user': DB_USER,
    'password': DB_PASSWORD,
    'driver': 'com.microsoft.sqlserver.jdbc.SQLServerDriver'
}

print(jdbc_url)

## 4. Criar e limpar bucket landing-zone

In [ ]:
s3_client = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

try:
    s3_client.head_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] ja existe.')
except Exception:
    s3_client.create_bucket(Bucket=LANDING_BUCKET)
    print(f'Bucket [{LANDING_BUCKET}] criado.')

response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
objetos = [{'Key': obj['Key']} for obj in response.get('Contents', [])]

if objetos:
    s3_client.delete_objects(Bucket=LANDING_BUCKET, Delete={'Objects': objetos})
    print(f'{len(objetos)} objeto(s) removido(s) do bucket [{LANDING_BUCKET}].')
else:
    print(f'Bucket [{LANDING_BUCKET}] ja estava vazio.')

## 5. Extrair tabelas do SQL Server e enviar CSVs para o MinIO

In [ ]:
resultados = []
tmp_root = Path(tempfile.mkdtemp(prefix='lojadb_csv_'))

try:
    for tabela in tabelas:
        print(f'Extraindo tabela dbo.{tabela}...')

        df = spark.read.jdbc(
            url=jdbc_url,
            table=f'dbo.{tabela}',
            properties=jdbc_properties
        )

        registros = df.count()
        output_dir = tmp_root / tabela

        (
            df.coalesce(1)
            .write
            .mode('overwrite')
            .option('header', True)
            .csv(str(output_dir))
        )

        part_file = next(output_dir.glob('part-*.csv'))
        s3_key = f'{tabela}.csv'

        s3_client.upload_file(
            Filename=str(part_file),
            Bucket=LANDING_BUCKET,
            Key=s3_key,
            ExtraArgs={'ContentType': 'text/csv'}
        )

        tamanho_kb = part_file.stat().st_size / 1024
        resultados.append({
            'tabela': tabela,
            'arquivo': s3_key,
            'registros': registros,
            'colunas': len(df.columns),
            'tamanho_kb': round(tamanho_kb, 1)
        })

        print(f'  {s3_key} enviado para s3://{LANDING_BUCKET}/{s3_key} ({registros} registros, {tamanho_kb:.1f} KB)')
finally:
    shutil.rmtree(tmp_root, ignore_errors=True)

print('Extracao concluida.')

## 6. Validacao do bucket landing-zone

In [ ]:
response = s3_client.list_objects_v2(Bucket=LANDING_BUCKET)
objetos = sorted(response.get('Contents', []), key=lambda obj: obj['Key'])

print(f'Arquivos no bucket [{LANDING_BUCKET}]:\n')
for obj in objetos:
    print(f'  {obj["Key"]:<25} {obj["Size"] / 1024:>8.1f} KB')

print('\nResumo da extracao:')
for item in resultados:
    print(f'  {item["tabela"]:<15} {item["registros"]:>6} registros | {item["colunas"]:>2} colunas | {item["tamanho_kb"]:>6.1f} KB')

## 7. Encerrar Spark

In [ ]:
spark.stop()
print('SparkSession finalizada.')